<a href="https://colab.research.google.com/github/cbonnin88/StatFlow/blob/main/StatFlow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install statsmodels

In [ ]:
import requests as re
import polars as pl
import numpy as np
import plotly.express as px
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

# **Extraction**
- Using dummyjson

In [ ]:
# Here, I used dummyjson since its a free public API that doesn't require authentication
api_url = 'https://dummyjson.com/carts?limit=500'

In [ ]:
def fetch_live_data():
  'Fetches live e-commcerce cart data via the GET request'
  print(f'Connection to {api_url}')

  # Send the request
  response = re.get(api_url,timeout=10)

  # Ensuring that I get got a successfull 200 OK response
  response.raise_for_status()

  # Parsing the JSON. The API wraps the data in a 'carts' key.
  json_payload = response.json()
  return json_payload['carts']

raw_json_data = fetch_live_data()
print(f'Successfully loaded {len(raw_json_data)} live e-commerce carts.')

Connection to https://dummyjson.com/carts?limit=500
Successfully loaded 208 live e-commerce carts.


# **Transformation**
- Using the Polars library

In [ ]:
df_raw = pl.DataFrame(raw_json_data, strict=False)

display(df_raw.head())

id,products,total,discountedTotal,userId,totalProducts,totalQuantity
i64,list[struct[8]],f64,f64,i64,i64,i64
1,"[{162,""Blue Frock"",29.99,4,119.96,12.13,105.41,""https://cdn.dummyjson.com/product-images/tops/blue-frock/thumbnail.webp""}, {113,""Generic Motorcycle"",3999.99,3,11999.97,12.1,10547.97,""https://cdn.dummyjson.com/product-images/motorcycle/generic-motorcycle/thumbnail.webp""}, … {138,""Baseball Ball"",8.99,2,17.98,1.71,17.67,""https://cdn.dummyjson.com/product-images/sports-accessories/baseball-ball/thumbnail.webp""}]",13037.88,11510.81,1,4,12
2,"[{86,""Man Short Sleeve Shirt"",19.99,5,99.95,6.83,93.12,""https://cdn.dummyjson.com/product-images/mens-shirts/man-short-sleeve-shirt/thumbnail.webp""}, {104,""Apple iPhone Charger"",19.99,2,39.98,18.52,32.58,""https://cdn.dummyjson.com/product-images/mobile-accessories/apple-iphone-charger/thumbnail.webp""}]",139.93,125.7,2,2,7
3,"[{24,""Fish Steak"",14.99,1,14.99,4.23,14.36,""https://cdn.dummyjson.com/product-images/groceries/fish-steak/thumbnail.webp""}, {123,""iPhone 13 Pro"",1099.99,1,1099.99,9.37,996.92,""https://cdn.dummyjson.com/product-images/smartphones/iphone-13-pro/thumbnail.webp""}, … {157,""Party Glasses"",19.99,5,99.95,11.22,88.74,""https://cdn.dummyjson.com/product-images/sunglasses/party-glasses/thumbnail.webp""}]",1794.85,1625.77,3,6,15
4,"[{92,""Sports Sneakers Off White Red"",109.99,3,329.97,0.04,329.84,""https://cdn.dummyjson.com/product-images/mens-shoes/sports-sneakers-off-white-red/thumbnail.webp""}, {8,""Dior J'adore"",89.99,4,359.96,14.72,306.97,""https://cdn.dummyjson.com/product-images/fragrances/dior-j'adore/thumbnail.webp""}]",689.93,636.81,4,2,7
5,"[{161,""Samsung Galaxy Tab White"",349.99,4,1399.96,18.2,1145.17,""https://cdn.dummyjson.com/product-images/tablets/samsung-galaxy-tab-white/thumbnail.webp""}, {39,""Soft Drinks"",1.99,4,7.96,17.48,6.57,""https://cdn.dummyjson.com/product-images/groceries/soft-drinks/thumbnail.webp""}, {3,""Powder Canister"",14.99,4,59.96,9.84,54.06,""https://cdn.dummyjson.com/product-images/beauty/powder-canister/thumbnail.webp""}]",1467.88,1205.8,5,3,12


In [ ]:
df_raw.describe()

statistic,id,products,total,discountedTotal,userId,totalProducts,totalQuantity
str,f64,f64,f64,f64,f64,f64,f64
"""count""",208.0,208.0,208.0,208.0,208.0,208.0,208.0
"""null_count""",0.0,0.0,0.0,0.0,0.0,0.0,0.0
"""mean""",104.5,null,18434.031875,16618.796058,104.5,3.846154,11.620192
"""std""",60.188592,null,34725.549339,31041.763712,60.188592,1.492456,5.425381
"""min""",1.0,null,6.97,6.65,1.0,2.0,2.0
"""25%""",53.0,null,405.93,360.45,53.0,2.0,7.0
"""50%""",105.0,null,2013.93,1674.54,105.0,4.0,12.0
"""75%""",156.0,null,17785.88,16911.51,156.0,5.0,16.0
"""max""",208.0,null,186110.85,155507.16,208.0,6.0,27.0


In [ ]:
rows = df_raw.shape[0]
columns = df_raw.shape[1]

print(f'Number of rows in the df_raw dataframe: {rows}')
print(f'Number of columns in the df_raw dataframe: {columns}')

Number of rows in the df_raw dataframe: 208
Number of columns in the df_raw dataframe: 7


In [ ]:
# Cleaning the data
df_ecommerce = (
    df_raw
    .with_columns([
        # Simulate A/B test assignment: Even user IDs to Control, Odd to Variant
        pl.when(pl.col('userId') % 2 ==0)
          .then(pl.lit('control'))
          .otherwise(pl.lit('variant'))
          .alias('group'),

        # Defining continuous metric: Using the 'discountedTotal' as Revenue
        pl.col('discountedTotal').cast(pl.Float64).alias('revenue'),

        # Defining binary metric: Did they convert to a 'large cart' ? (more than 4 items)
        pl.when(pl.col('totalProducts') >=4)
          .then(pl.lit(1))
          .otherwise(pl.lit(0))
          .cast(pl.Int32)
          .alias('converted')
    ])
    # Selecting only the columns that I need for the analysis to keepy memory footprint tiny
    .select(['id','userId','group','converted','revenue'])
)

In [ ]:
print('--- Cleaned & Transformed Data ---')
display(df_ecommerce.head())

--- Cleaned & Transformed Data ---


id,userId,group,converted,revenue
i64,i64,str,i32,f64
1,1,"""variant""",1,11510.81
2,2,"""control""",0,125.7
3,3,"""variant""",1,1625.77
4,4,"""control""",0,636.81
5,5,"""variant""",0,1205.8


# **Analysis**
- SciPy & Statsmodels

In [ ]:
def calc_conversion_metrics(df_pl, group_col='group', target_col='converted'):
  # Calculates Z-Test for binary conversion rate.
  summary = df_pl.group_by(group_col).agg([
      pl.len().alias('visitors'),
      pl.col(target_col).sum().alias('conversions')
  ]).to_pandas()

  summary.set_index(group_col, inplace=True)
  control, variant = summary.loc['control'],summary.loc['variant']

  successes = np.array([variant['conversions'], control['conversions']])
  trials = np.array([variant['visitors'], control['visitors']])

  z_stats, p_value = proportions_ztest(successes, trials)

  return {
      'Metrics': 'Large Cart Conversion Rate',
      'Control': f'{control['conversions']/control['visitors']:.2%}',
      'Variant': f'{variant['conversions']/variant['visitors']:.2%}',
      'P-Value': round(p_value,4),
      'Significant (95%)': p_value < 0.05
  }

def calc_continuous_metrics(df_pl, group_col='group',target_col='revenue'):
  # Calculatings Welch's T-Test for continuous data using SciPy
  control_data = df_pl.filter(pl.col(group_col) == 'control')[target_col].to_numpy()
  variant_data = df_pl.filter(pl.col(group_col) == 'variant')[target_col].to_numpy()

  # Welch's T-Test
  t_stat, p_value = stats.ttest_ind(variant_data,control_data,equal_var=False)

  return {
      'Metric': 'Average Cart Revenue',
      'Control': f'€{np.mean(control_data):.2f}',
      'Variant': f'€{np.mean(variant_data):.2f}',
      'P-Value': round(p_value,4),
      'Significant (95%)': p_value < 0.05
  }

In [ ]:
print('--- A/B Test Results ---')
display(calc_conversion_metrics(df_ecommerce))


--- A/B Test Results ---


{'Metrics': 'Large Cart Conversion Rate',
 'Control': '48.08%',
 'Variant': '61.54%',
 'P-Value': np.float64(0.0511),
 'Significant (95%)': np.False_}

In [ ]:
display(calc_continuous_metrics(df_ecommerce))

{'Metric': 'Average Cart Revenue',
 'Control': '€14993.94',
 'Variant': '€18243.65',
 'P-Value': np.float64(0.4516),
 'Significant (95%)': np.False_}

# **Visualization**
- Using Plotly

In [ ]:
# 1. Bar Chart: Conversion Rate
conv_df = df_ecommerce.group_by('group').agg(
    pl.col('converted').mean().alias('Conversion Rate')
).to_pandas()

display(conv_df)

,group,Conversion Rate
0,control,0.480769
1,variant,0.615385


In [ ]:
fig_conversion = px.bar(
    conv_df,
    x='group',
    y='Conversion Rate',
    color='group',
    title='Large Cart Conversion Rate by Test Group',
    labels= {'group':'Test Group','control':'Control','variant':'Variant'},
    text_auto='.2%',
    color_discrete_sequence=px.colors.sequential.Viridis_r
)

fig_conversion.update_layout(showlegend=False, yaxis_tickformat='.0%')
fig_conversion.show()

In [ ]:
# 2. Revenue Distribution
fig_revenue = px.box(
    df_ecommerce.to_pandas(),
    x='group',
    y='revenue',
    color='group',
    title = 'E-Commerce Revenue Distribution per Cart',
    labels = {'revenue':'Cart Revenue {€}','group':'Test Group'},
    color_discrete_sequence = px.colors.sequential.Viridis_r
)

fig_revenue.update_layout(showlegend=False)
fig_revenue.show()